# Preprocessing


## Important please download data from https://data.police.uk/data/ into the folder named raw before starting to run the code 
Also please keep in mind this sheet starts of using a small sample of data to clean before cleaning all data properly

## 1) Import

In [26]:
import os , re, shutil
import pandas as pd
import numpy as np

## 3) Inspect each Street sample (shape, columns, info, missing, duplicates)

In [5]:
# --- City of London Street ---
print('City of London Street:', col_street.shape); print(list(col_street.columns))
print(col_street.info())
print('Missing:', col_street.isna().sum().sort_values(ascending=False).head(10))
print('Dups:', col_street.duplicated().sum())

# --- Leicestershire Street ---
print('\nLeicestershire Street:', lei_street.shape); print(list(lei_street.columns))
print(lei_street.info())
print('Missing:', lei_street.isna().sum().sort_values(ascending=False).head(10))
print('Dups:', lei_street.duplicated().sum())

# --- Metropolitan Street ---
print('\nMetropolitan Street:', met_street.shape); print(list(met_street.columns))
print(met_street.info())
print('Missing:', met_street.isna().sum().sort_values(ascending=False).head(10))
print('Dups:', met_street.duplicated().sum())

# --- West Midlands Street ---
print('\nWest Midlands Street:', wm_street.shape); print(list(wm_street.columns))
print(wm_street.info())
print('Missing:', wm_street.isna().sum().sort_values(ascending=False).head(10))
print('Dups:', wm_street.duplicated().sum())

City of London Street: (719, 12)
['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 719 entries, 0 to 718
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               663 non-null    object 
 1   Month                  719 non-null    object 
 2   Reported by            719 non-null    object 
 3   Falls within           719 non-null    object 
 4   Longitude              674 non-null    float64
 5   Latitude               674 non-null    float64
 6   Location               719 non-null    object 
 7   LSOA code              674 non-null    object 
 8   LSOA name              674 non-null    object 
 9   Crime type             719 non-null    object 
 10  Last outcome category  663 non-null    object 
 11  Context        

We preview **shape, columns, info, missing, duplicates** for each force before combining.

## 4) Standardise column names (lowercase, underscores)

In [6]:
col_outcomes.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Outcome type
0,d4a960e9d2388010b23d0fbdef9f53a27f7c668b75a551...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Investigation complete; no suspect identified
1,f6aaf45a07edc72c86fd887633654890409b71c5dc1338...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Formal action is not in the public interest
2,5c0c59621de4075dbd5ff34ef816cafd7a229261cebf00...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Investigation complete; no suspect identified
3,b0d3fee479ba6a0a594fea132d9aa5c8dfd8022d4901a8...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Formal action is not in the public interest
4,657835b9aa841a17a35a4970ef29311baa229fa00a46fb...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Unable to prosecute suspect


In [7]:
for df in [col_street, lei_street, met_street, wm_street, col_outcomes, lei_outcomes, met_outcomes, wm_outcomes]:
    df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_'))

# Optional presence check on a representative frame
example_cols = ['crime_id','month','reported_by','falls_within','longitude','latitude','location','lsoa_code','lsoa_name','crime_type']
print('Presence check (City of London street):')
for c in example_cols:
    print(f'{c:>15}:', c in col_street.columns)

Presence check (City of London street):
       crime_id: True
          month: True
    reported_by: True
   falls_within: True
      longitude: True
       latitude: True
       location: True
      lsoa_code: True
      lsoa_name: True
     crime_type: True


We keep **`reported_by`** as the force identifier. We’ll drop `falls_within` later as redundant (if present).

## 5) combine Street and Outcomes across forces

In [8]:
# --- Combine Street data ( ---
street_combined = pd.concat(
    [col_street, lei_street, met_street, wm_street],
    axis=0, ignore_index=True
)

# --- Combine Outcomes data 
outcomes_combined = pd.concat(
    [col_outcomes, lei_outcomes, met_outcomes, wm_outcomes],
    axis=0, ignore_index=True
)

print('street_combined:', street_combined.shape)
print('outcomes_combined:', outcomes_combined.shape)
street_combined.head(3)

street_combined: (127543, 12)
outcomes_combined: (110910, 10)


,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,context
0,4a14d4745da0a2219ecf913fdc28a0c84ae8408954cac6...,2023-01,City of London Police,City of London Police,-0.10622,51.518275,On or near B500,E01000916,Camden 027B,Other theft,Status update unavailable,NaN
1,e6e32581c99c5b0f46e5a4d6343e213df349d7069e847d...,2023-01,City of London Police,City of London Police,-0.10701,51.521480,On or near Saffron Street,E01000916,Camden 027B,Other crime,Status update unavailable,NaN
2,7b7cb8e7debe8b0ec1637e7cb1dad832cea4eba16c5f52...,2023-01,City of London Police,City of London Police,-0.11035,51.518090,On or near Holborn,E01000917,Camden 027C,Theft from the person,Investigation complete; no suspect identified,NaN


## 6) Inspect combined frames (shape, columns, sample, missing)

In [9]:
print('Combined Street shape:', street_combined.shape)
print('Combined Street columns:', list(street_combined.columns))
print('\nMissing (Street combined):')
print(street_combined.isna().sum().sort_values(ascending=False).head(15))

print('\nCombined Outcomes shape:', outcomes_combined.shape)
print('Combined Outcomes columns:', list(outcomes_combined.columns))
print('\nMissing (Outcomes combined):')
print(outcomes_combined.isna().sum().sort_values(ascending=False).head(15))

display(street_combined.head(5))
display(outcomes_combined.head(5))

Combined Street shape: (127543, 12)
Combined Street columns: ['crime_id', 'month', 'reported_by', 'falls_within', 'longitude', 'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type', 'last_outcome_category', 'context']

Missing (Street combined):
context                  127543
crime_id                  16526
last_outcome_category     16526
longitude                  2421
lsoa_name                  2421
lsoa_code                  2421
latitude                   2421
month                         0
falls_within                  0
reported_by                   0
location                      0
crime_type                    0
dtype: int64

Combined Outcomes shape: (110910, 10)
Combined Outcomes columns: ['crime_id', 'month', 'reported_by', 'falls_within', 'longitude', 'latitude', 'location', 'lsoa_code', 'lsoa_name', 'outcome_type']

Missing (Outcomes combined):
longitude       2702
latitude        2702
lsoa_name       2702
lsoa_code       2702
month              0
crime_id       

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,context
0,4a14d4745da0a2219ecf913fdc28a0c84ae8408954cac6...,2023-01,City of London Police,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Other theft,Status update unavailable,NaN
1,e6e32581c99c5b0f46e5a4d6343e213df349d7069e847d...,2023-01,City of London Police,City of London Police,-0.107010,51.521480,On or near Saffron Street,E01000916,Camden 027B,Other crime,Status update unavailable,NaN
2,7b7cb8e7debe8b0ec1637e7cb1dad832cea4eba16c5f52...,2023-01,City of London Police,City of London Police,-0.110350,51.518090,On or near Holborn,E01000917,Camden 027C,Theft from the person,Investigation complete; no suspect identified,NaN
3,f7fc44e1e76332f0f575b788522329e6f3ce566fd7472d...,2023-01,City of London Police,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other crime,Status update unavailable,NaN
4,8083dafd1770af1afca2320c13cbae2420bd2877d9ef29...,2023-01,City of London Police,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Status update unavailable,NaN


,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,outcome_type
0,d4a960e9d2388010b23d0fbdef9f53a27f7c668b75a551...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Investigation complete; no suspect identified
1,f6aaf45a07edc72c86fd887633654890409b71c5dc1338...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Formal action is not in the public interest
2,5c0c59621de4075dbd5ff34ef816cafd7a229261cebf00...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Investigation complete; no suspect identified
3,b0d3fee479ba6a0a594fea132d9aa5c8dfd8022d4901a8...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Formal action is not in the public interest
4,657835b9aa841a17a35a4970ef29311baa229fa00a46fb...,2023-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Unable to prosecute suspect


In [10]:
street_combined['crime_id'].isin(outcomes_combined['crime_id']).mean()


np.float64(0.41082615274848483)

In [11]:
data_copy= street_combined.copy()
data_copy2 = outcomes_combined.copy()

## 7) Types & light cleaning 

In [12]:
street_combined.drop(columns = ['context'], inplace = True)
# first I removed Context as it has all Nan values
#Below i ahve converted long/lat/month to correct datatypes
if 'month' in street_combined.columns:
    street_combined['month'] = pd.to_datetime(street_combined['month'])
if 'month' in outcomes_combined.columns:
    outcomes_combined['month'] = pd.to_datetime(outcomes_combined['month'])

for col in ['longitude','latitude']:
    if col in street_combined.columns:
        street_combined[col] = pd.to_numeric(street_combined[col])

#removed duplicates
before = len(street_combined)
street_combined = street_combined.drop_duplicates()
print('Removed duplicates (Street combined):', before - len(street_combined))




#dropped falls within if repretd by is there as well because it repeats same data. safety check
to_drop = []
if 'reported_by' in street_combined.columns and 'falls_within' in street_combined.columns:
    to_drop.append('falls_within')
if to_drop:
    print('Dropping columns:', to_drop)
    street_combined = street_combined.drop(columns=to_drop)

street_combined.head(3)

Removed duplicates (Street combined): 7119
Dropping columns: ['falls_within']


,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
0,4a14d4745da0a2219ecf913fdc28a0c84ae8408954cac6...,2023-01-01,City of London Police,-0.10622,51.518275,On or near B500,E01000916,Camden 027B,Other theft,Status update unavailable
1,e6e32581c99c5b0f46e5a4d6343e213df349d7069e847d...,2023-01-01,City of London Police,-0.10701,51.521480,On or near Saffron Street,E01000916,Camden 027B,Other crime,Status update unavailable
2,7b7cb8e7debe8b0ec1637e7cb1dad832cea4eba16c5f52...,2023-01-01,City of London Police,-0.11035,51.518090,On or near Holborn,E01000917,Camden 027C,Theft from the person,Investigation complete; no suspect identified


## 8) Create views + merge Outcomes (left join on keys)

In [13]:
full_sample = street_combined.copy()
sample_id =street_combined.dropna(subset=['crime_id']).copy()

print('full_sample:', full_sample.shape)
print('sample_id  :', sample_id.shape)




outcomes_uniq = outcomes_combined.dropna(subset=['crime_id']).drop_duplicates(subset=['crime_id']).copy()


merged = pd.merge(full_sample, outcomes_uniq, how='left', on='crime_id', suffixes=('', '_outcome'))

print('Merged shape:', merged.shape)
merged.head(5)

full_sample: (120424, 10)
sample_id  : (110409, 10)
Merged shape: (120424, 19)


,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,month_outcome,reported_by_outcome,falls_within,longitude_outcome,latitude_outcome,location_outcome,lsoa_code_outcome,lsoa_name_outcome,outcome_type
0,4a14d4745da0a2219ecf913fdc28a0c84ae8408954cac6...,2023-01-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Other theft,Status update unavailable,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,e6e32581c99c5b0f46e5a4d6343e213df349d7069e847d...,2023-01-01,City of London Police,-0.107010,51.521480,On or near Saffron Street,E01000916,Camden 027B,Other crime,Status update unavailable,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7b7cb8e7debe8b0ec1637e7cb1dad832cea4eba16c5f52...,2023-01-01,City of London Police,-0.110350,51.518090,On or near Holborn,E01000917,Camden 027C,Theft from the person,Investigation complete; no suspect identified,2023-01-01,City of London Police,City of London Police,-0.11035,51.51809,On or near Holborn,E01000917,Camden 027C,Investigation complete; no suspect identified
3,f7fc44e1e76332f0f575b788522329e6f3ce566fd7472d...,2023-01-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other crime,Status update unavailable,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8083dafd1770af1afca2320c13cbae2420bd2877d9ef29...,2023-01-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Status update unavailable,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9) Quick checks + save combined outputs

In [23]:
report = {
    'rows_street_combined' : len(full_sample),
    'rows_outcomes_combined': len(outcomes_combined),
    'rows_merged' : len(merged),
    'pct_missing_coords'   : float(((full_sample['latitude'].isna() | full_sample['longitude'].isna()).mean()*100)) if set(['latitude','longitude']).issubset(full_sample.columns) else None
}
print('Report:', report)

# Save

os.makedirs('../data/preprocessed', exist_ok=True)
full_sample.to_csv('../data/preprocessed/MULTI_FORCES_street_clean.csv', index=False)
merged.to_csv('../data/preprocessed/MULTI_FORCES_street_outcomes_merged.csv', index=False)
print('Saved: ../data/preprocessed/MULTI_FORCES_street_clean.csv')
print('Saved: ../data/preprocessed/MULTI_FORCES_street_outcomes_merged.csv')

Report: {'rows_street_combined': 120424, 'rows_outcomes_combined': 110910, 'rows_merged': 120424, 'pct_missing_coords': 2.0037534046369494}
Saved: ../data/preprocessed/MULTI_FORCES_street_clean.csv
Saved: ../data/preprocessed/MULTI_FORCES_street_outcomes_merged.csv


# Lets delete
This first section I used a small part of the data as a sample to clean and check before doing the whole dataset next I will create functions that will be able to handle all data The next section of code will delete the saved sample data as it will not be needed

In [14]:
#run this to delete smaple files this code is for testing 
import os, shutil

# Path to your preprocessed folder
pre_path = r"C:\Users\micha\Documents\PyData_Project\data\preprocessed"

# Delete all files inside (but keep the folder itself)
for f in os.listdir(pre_path):
    file_path = os.path.join(pre_path, f)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # delete file or link
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # delete subfolder
    except Exception as e:
        print(f" Failed to delete {file_path}: {e}")

print(" All files deleted from:", pre_path)



 All files deleted from: C:\Users\micha\Documents\PyData_Project\data\preprocessed


# 10) Functions
Now we will write function that can do this preprocessing so we can do it on a large scale

### Load files

In [15]:
BASE_PATH = r"C:\Users\micha\Documents\PyData_Project"
def load_force_data(month_str, force, base_path=BASE_PATH):
    """
    month_str: 'YYYY-MM' so essntialy the files have year-month as part of name,i did this to make geeting the filename easier ill use as a variable 
    force: one of ['city-of-london','leicestershire','metropolitan','west-midlands']
    Returns: (street_df, outcomes_df)
    """
    folder = os.path.join(base_path, "data", "raw", month_str)
    street_file   = f"{month_str}-{force}-street.csv"
    outcomes_file = f"{month_str}-{force}-outcomes.csv"

    street_df   = pd.read_csv(os.path.join(folder, street_file))
    outcomes_df = pd.read_csv(os.path.join(folder, outcomes_file))
    return street_df, outcomes_df


### Clean files

In [16]:
def clean_street_data(df):
    """Standardise columns, convert types, dedupe, strip text, drop redundant 'falls_within'."""
    # standardise col names
    df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_'))
    df.drop(columns = ['context'], inplace = True)

    # types
    if 'month' in df.columns:
        df['month'] = pd.to_datetime(df['month'], errors='coerce')
    for col in ['longitude','latitude']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # dedupe
    df = df.drop_duplicates()

    

    # keep reported_by; drop falls_within if both exist
    if ('reported_by' in df.columns) and ('falls_within' in df.columns):
        df = df.drop(columns=['falls_within'])

    return df


### Combine Files

In [17]:
def concat_forces(force_dfs):
    """
    Combine multiple DataFramesusing pd.concat.
    Returns a single DataFrame
    
    
    """
    combined = pd.concat(force_dfs, axis=0, ignore_index=True)
    print(" Combined shape:", combined.shape)
    return combined


### Merge with outcomes

In [18]:
def merge_with_outcomes(street_df, outcomes_df):
    """Left join outcomes on 'crime_id' only.  drop null crime_id on right."""
    # standardise 
    outcomes_df.columns = (outcomes_df.columns.str.strip().str.lower().str.replace(' ', '_'))

    # ensure month is datetime 
    if 'month' in outcomes_df.columns:
        outcomes_df['month'] = pd.to_datetime(outcomes_df['month'], errors='coerce')

    # keep only valid IDs and de-dup by 'crime_id'
    outcomes_uniq = (
        outcomes_df
        .dropna(subset=['crime_id'])
        .drop_duplicates(subset=['crime_id'])
        .copy()
    )

    merged = pd.merge(
        street_df, outcomes_uniq,
        how='left', on='crime_id',
        suffixes=('', '_outcome')
    )

    # if outcomes has duplicate columns, drop them 
    for c in ['longitude_outcome','latitude_outcome']:
        if c in merged.columns:
            merged = merged.drop(columns=c)

    return merged


### Save Cleaned 

In [19]:
def save_cleaned_data(street_clean, merged, label, base_path=BASE_PATH):
    """
    Save both the cleaned Street data and merged (Street + Outcomes) data.
    
    """
    out_dir = os.path.join(base_path, "data", "preprocessed")
    os.makedirs(out_dir, exist_ok=True)

    path_clean = os.path.join(out_dir, f"{label}_street_clean.csv")
    path_merged = os.path.join(out_dir, f"{label}_street_outcomes_merged.csv")

    street_clean.to_csv(path_clean, index=False)
    merged.to_csv(path_merged, index=False)

    print(f" Saved: {os.path.basename(path_clean)} and {os.path.basename(path_merged)}")


### Run it 

In [25]:
import os

BASE_PATH = r"C:\Users\micha\Documents\PyData_Project"
forces = ["city-of-london", "leicestershire", "metropolitan", "west-midlands"]

# list of months
months = [
    "2023-01","2023-02","2023-03","2023-04","2023-05","2023-06",
    "2023-07","2023-08","2023-09","2023-10","2023-11","2023-12",
    "2024-01","2024-02","2024-03","2024-04","2024-05","2024-06",
    "2024-07","2024-08","2024-09","2024-10","2024-11","2024-12"
]

all_street, all_merged = [], []

for month_str in months:
    clean_list, merged_list = [], []

    for f in forces:
        s_raw, o_raw = load_force_data(month_str, f)#load data
        s_clean = clean_street_data(s_raw)#run clean function
        m = merge_with_outcomes(s_clean, o_raw)
        #save_cleaned_data(s_clean, m, f"{f}-{month_str}") saves too many so commented out but if you want to save monthly as well can do
        clean_list.append(s_clean)
        merged_list.append(m)

    # Combine per month
    street_combined = concat_forces(clean_list)
    merged_combined = concat_forces(merged_list)
    #save_cleaned_data(street_combined, merged_combined, f"multi-forces-{month_str}")  - this will save by month if wanted

    # Add to overall lists for 2-year dataset
    all_street.append(street_combined)
    all_merged.append(merged_combined)

# Combine across all months (2 years total)
street_full = concat_forces(all_street)
merged_full = concat_forces(all_merged)
save_cleaned_data(street_full, merged_full, "multi-forces-2023-2024")

print(" Full combined shape:", street_full.shape, merged_full.shape)


 Combined shape: (120424, 10)
 Combined shape: (120424, 17)
 Combined shape: (115229, 10)
 Combined shape: (115229, 17)
 Combined shape: (127377, 10)
 Combined shape: (127377, 17)
 Combined shape: (120283, 10)
 Combined shape: (120283, 17)
 Combined shape: (127222, 10)
 Combined shape: (127222, 17)
 Combined shape: (136337, 10)
 Combined shape: (136337, 17)
 Combined shape: (130007, 10)
 Combined shape: (130007, 17)
 Combined shape: (123775, 10)
 Combined shape: (123775, 17)
 Combined shape: (124986, 10)
 Combined shape: (124986, 17)
 Combined shape: (133089, 10)
 Combined shape: (133089, 17)
 Combined shape: (125271, 10)
 Combined shape: (125271, 17)
 Combined shape: (122206, 10)
 Combined shape: (122206, 17)
 Combined shape: (119339, 10)
 Combined shape: (119339, 17)
 Combined shape: (116533, 10)
 Combined shape: (116533, 17)
 Combined shape: (120796, 10)
 Combined shape: (120796, 17)
 Combined shape: (119797, 10)
 Combined shape: (119797, 17)
 Combined shape: (126991, 10)
 Combined 

### Dont run this below

In [24]:
import os, shutil

# Path to your preprocessed folder
pre_path = r"C:\Users\micha\Documents\PyData_Project\data\preprocessed"

# Delete all files inside (but keep the folder itself)
for f in os.listdir(pre_path):
    file_path = os.path.join(pre_path, f)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # delete file or link
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # delete subfolder
    except Exception as e:
        print(f" Failed to delete {file_path}: {e}")

print(" All files deleted from:", pre_path)


 All files deleted from: C:\Users\micha\Documents\PyData_Project\data\preprocessed


### Great we have our file cleaned and combined together pleae go to my EDA file to see what i did next :)
I have saved a lot of files but i will make it very clear wich file is used for the next part
also the files we will be using will be saved into a seperate folder

In [22]:
import os
street_path = os.path.join(
    BASE_PATH, "data", "preprocessed", "multi-forces-2023-2024_street_clean.csv"
)
street_full = pd.read_csv(street_path)


street_full.shape
street_full.tail()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
2987248,50543f37c50efd377e73426b1955801475e19e7211e14c...,2024-12-01,West Midlands Police,-2.119472,52.572273,On or near Silver Birch Road,E01034315,Wolverhampton 035J,Vehicle crime,Investigation complete; no suspect identified
2987249,7895ba9d356d0ccd23f2db9aa38595934cec2a4e0b5065...,2024-12-01,West Midlands Police,-2.116945,52.570630,On or near Beech Tree Road,E01034315,Wolverhampton 035J,Violence and sexual offences,Unable to prosecute suspect
2987250,9149874c47a05385a0924528d9399474a82cb49b273902...,2024-12-01,West Midlands Police,-2.118019,52.569505,On or near Thompson Avenue,E01034315,Wolverhampton 035J,Violence and sexual offences,Status update unavailable
2987251,a6116a1831bd5de9ab6e4540b1f92c8c4684b4bd8dc61f...,2024-12-01,West Midlands Police,-2.119472,52.572273,On or near Silver Birch Road,E01034315,Wolverhampton 035J,Violence and sexual offences,Action to be taken by another organisation
2987252,937b6f5b0ba3fdc77803dead97ac4dbfff22a3790a2f5f...,2024-12-01,West Midlands Police,-2.119472,52.572273,On or near Silver Birch Road,E01034315,Wolverhampton 035J,Violence and sexual offences,Unable to prosecute suspect
